In [56]:
import numpy as np
import pandas as pd

In [57]:
from datasets import load_dataset

dataset = load_dataset("go_emotions", "simplified")

In [58]:
data = pd.read_csv("final_dataset.xls")
data["category"].value_counts()

category
Distress     29521
Neutral      17850
Offensive     7792
Emergency     4000
Name: count, dtype: int64

In [59]:
hard_repeated = pd.concat([hard_examples]*10)

new_data = pd.concat([data, hard_repeated]).sample(frac=1).reset_index(drop=True)

In [60]:
new_data["label"] = new_data["category"].map({"Distress": 0, "Offensive": 1, "Emergency": 2, "Neutral": 3}).astype(int)

In [ ]:
new_data

,text,category,label
0,More like 25% in this scenario. Precarious tho...,Neutral,3
1,"If it's a real challenge for you, u can help y...",Distress,0
2,"Omg, maybe best plot twist. I wish that will h...",Distress,0
3,Yes please go get tested even if you don’t hav...,Distress,0
4,I brought my love a cherrrryyyyy....,Distress,0
...,...,...,...
59258,"Oh my bad, i didn't know, thanks for telling me!",Distress,0
59259,The irony here of course is that your degree b...,Distress,0
59260,As long as you’re not being manipulated into h...,Neutral,3
59261,"Interesting. Even if most disagree with it, I'...",Distress,0


In [62]:
from datasets import ClassLabel

class_label = ClassLabel(
    num_classes=4,
    names=["Distress", "Offensive", "Emergency", "Neutral"]
)

In [63]:
from datasets import Dataset

dataset = Dataset.from_pandas(new_data)

dataset = dataset.cast_column("label", class_label)

Casting the dataset: 100%|██████████| 59263/59263 [00:00<00:00, 6576717.50 examples/s]


In [64]:
dataset = dataset.train_test_split(
    test_size=0.2,
    stratify_by_column="label"
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [65]:
train_dataset

Dataset({
    features: ['text', 'category', 'label'],
    num_rows: 47410
})

In [66]:
# !pip install scikit-learn

In [67]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch

train_labels = train_dataset["label"]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([0.5019, 1.9016, 3.6582, 0.8277])


In [68]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mobilebert-uncased")

def tokenize_function(examples):
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    tokens["labels"] = examples["label"]
    return tokens

tokenized_datasets = train_dataset.map(tokenize_function, batched=True)
tokenized_datasets_test = test_dataset.map(tokenize_function, batched=True)

tokenized_datasets.set_format("torch")
tokenized_datasets_test.set_format("torch")
print("Train Set: " ,tokenized_datasets)
print("Test Set: " ,tokenized_datasets_test)

Map: 100%|██████████| 11853/11853 [00:01<00:00, 11015.44 examples/s]

Train Set:  Dataset({
    features: ['text', 'category', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 47410
})
Test Set:  Dataset({
    features: ['text', 'category', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 11853
})


In [69]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("google/mobilebert-uncased", num_labels=4, problem_type="single_label_classification")

Loading weights: 100%|██████████| 1111/1111 [00:01<00:00, 846.38it/s, Materializing param=mobilebert.encoder.layer.23.output.dense.weight]                  
MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED

In [70]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [71]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./jarvis",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    eval_dataset=tokenized_datasets_test,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

In [72]:
trainer.train()
eval_results = trainer.evaluate()
eval_results

Epoch,Training Loss,Validation Loss
1,0.738485,0.749209
2,0.601693,0.685321
3,0.466794,0.761370


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.24it/s]
There were missing keys in the checkpoint model loaded: ['mobilebert.embeddings.LayerNorm.bias', 'mobilebert.embeddings.LayerNorm.weight', 'mobilebert.encoder.layer.0.attention.output.LayerNorm.bias', 'mobilebert.encoder.layer.0.attention.output.LayerNorm.weight', 'mobilebert.encoder.layer.0.output.LayerNorm.bias', 'mobilebert.encoder.layer.0.output.LayerNorm.weight', 'mobilebert.encoder.layer.0.output.bottleneck.LayerNorm.bias', 'mobilebert.encoder.layer.0.output.bottleneck.LayerNorm.weight', 'mobilebert.encoder.layer.0.bottleneck.input.LayerNorm.bias', 'mobilebert.encoder.layer.0.bottleneck.input.LayerNorm.weight', 'mobilebert.encoder.layer.0.bottleneck.attention.LayerNorm.bias', 'mobilebert.encoder.layer.0.bottleneck.attention.LayerNorm.weight', 'mobilebert.encoder.layer.0.ffn.0.output.LayerNorm.bias', 'mobilebert.encoder.layer.0.ffn.0.output.LayerNorm.weight', 'mobilebert.encoder.layer.0.ffn.1.output.LayerNorm.bias

{'eval_loss': 0.6908902525901794,
 'eval_runtime': 81.3344,
 'eval_samples_per_second': 145.732,
 'eval_steps_per_second': 18.221,
 'epoch': 3.0}

In [1]:
trainer.save_model("jarvis")
tokenizer.save_pretrained("jarvis")

NameError: name 'trainer' is not defined

In [2]:
from transformers import pipeline,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./jarvis")

predictor = pipeline("text-classification", model="./jarvis",tokenizer=tokenizer)
text_arr = ["I don't see any reason to continue anymore.",
    "I might harm myself today.",
    "And then you're stuck on that medication for life, sounds like a scary trade-off.",
    "He probably misread it as *gunst*, he didn't come here for gunst.",]

tricky_texts = [
    "lol i literally want to disappear forever",
    "I'm gonna kill this exam tomorrow!",
    "This traffic is killing me",
    "I think I may have taken too many pills",
    "Oh great, another perfect day to feel nothing",
    "I'm fine. totally fine. everything is fine.",
]

results = predictor(tricky_texts)
results

c:\Users\Atharva Nair\OneDrive\Desktop\mlproject\bert_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1113/1113 [00:01<00:00, 741.06it/s, Materializing param=mobilebert.encoder.layer.23.output.dense.weight]                  


[{'label': 'LABEL_0', 'score': 0.9839330911636353},
 {'label': 'LABEL_3', 'score': 0.8697584271430969},
 {'label': 'LABEL_3', 'score': 0.9587551355361938},
 {'label': 'LABEL_2', 'score': 0.9990578293800354},
 {'label': 'LABEL_0', 'score': 0.9727640151977539},
 {'label': 'LABEL_0', 'score': 0.9380171895027161}]

In [3]:
from sklearn.metrics import (accuracy_score, classification_report,confusion_matrix, roc_auc_score)

true_labels = ["LABEL_0", "LABEL_3", "LABEL_3", "LABEL_2", "LABEL_0", "LABEL_0"]
pred_labels = [out["label"] for out in results] 

In [4]:
print(int(accuracy_score(true_labels,pred_labels)*100))

100


In [ ]:
label_mapping = {
    'LABEL_0': "Distress", 
    'LABEL_1': "Offensive", 
    'LABEL_2': "Emergency", 
    'LABEL_3': "Neutral"   
}
pred_labels_names = [label_mapping[p] for p in pred_labels]

test_df = test_dataset["text"]
test_labels = test_dataset["category"]

errors = []
for text, true, pred, result in zip(test_df, test_labels, pred_labels_names, results):
    if true!=pred:
        errors.append({
            "text":         text,
            "true_label":   true,         
            "predicted_as": pred,        
            "confidence":   result["score"]
        })

error_df = pd.DataFrame(errors)

In [36]:
print(len(test_df))
print(len(errors), error_df["true_label"].value_counts())

11833
4 true_label
Distress    4
Name: count, dtype: int64
